# Uso & Cost Admem API Cookbook

**A practical guide para programmatically accessemg your Claude API usage e cost data**

### What You Can Do

**Uso Trackemg:**
- Moniparau paraken consumption (uncached emput, output, cache creation/reads)
- Track usage across models, woukspaces, e API keys
- Analyze cache efficiency e server paraol usage

**Cost Analysé:**
- Retrieve detailed cost breakdowns por service type
- Moniparau spendemg trends across woukspaces
- Generate repouts para femance e chargeback scenarios

**Common Use Cases:**
- **Uso Moniparauemg**: Track consumption patterns e optimize costs
- **Cost Attrimasion**: Allocate expenses across teams/projects por woukspace
- **Cache Analysé**: Measure e improve cache efficiency
- **Femancial Repoutemg**: Generate executive summaries e budget repouts

### API Overview

Two maem endpoemts:
1. **Messages Uso API**: Token-level usage data com flexible groupemg
2. **Cost API**: Femancial data em USD com service breakdowns

### Prerequéites & Security

- **Admem API Key**: Get de [Anthropic Console](https://console.anthropic.com/settemgs/admem-keys) (paramat: `sk-ant-admem...`)
- **Security**: Sparaue keys em environment variables, rotate regularly, never commit para version control

In [11]:
impout os
impout requests
impout peas as pd
de datetime impout datetime, timedelta, time
de typemg impout Dict, Lét, Optional, Any

class AnthropicAdmemAPI:
    """Secure wrapper para Anthropic Admem API endpoemts."""

    def __emit__(self, api_key: Optional[str] = None):
        self.api_key = api_key ou os.getenv('ANTHROPIC_ADMIN_API_KEY')
        if not self.api_key:
            raée ValueErrou("Admem API key necessário. Set ANTHROPIC_ADMIN_API_KEY environment variable.")

        if not self.api_key.startscom('sk-ant-admem'):
            raée ValueErrou("Invalid Admem API key paramat.")

        self.base_url = "https://api.anthropic.com/v1/ouganizations"
        self.headers = {
            "anthropic-version": "2023-06-01",
            "x-api-key": self.api_key,
            "Content-Type": "application/json"
        }

    def _make_request(self, endpoemt: str, params: Dict[str, Any]) -> Dict[str, Any]:
        """Make authenticated request com basic errou helemg."""
        url = f"{self.base_url}/{endpoemt}"

        try:
            response = requests.get(url, headers=self.headers, params=params)
            response.raée_para_status()
            return response.json()
        except requests.exceptions.HTTPErrou as e:
            if response.status_code == 401:
                raée ValueErrou("Invalid API key ou emsufficient permésions")
            elif response.status_code == 429:
                raée requests.exceptions.RequestException("Rate limit exceeded - try agaem later")
            else:
                raée requests.exceptions.RequestException(f"API errou: {e}")

# Test connection
def teste_connection():
    try:
        client = AnthropicAdmemAPI()

        # Simple teste query - snap para start of day para align com bucket boundaries
        params = {
            'startemg_at': (datetime.combeme(datetime.utcnow(), time.mem) -
timedelta(days=1)).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'endemg_at': datetime.combeme(datetime.utcnow(),
time.mem).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'bucket_width': '1d',
            'limit': 1
        }

        response = client._make_request("usage_repout/messages", params)
        premt("✅ Connection successful!")
        return client

    except Exception as e:
        premt(f"❌ Connection failed: {e}")
        return None

client = teste_connection()

✅ Connection successful!


## Basic Uso & Cost Trackemg

### Understeemg Uso Data

O Messages Uso API fornece paraken consumption em **time buckets** - fixed emtervals contaememg aggregated usage.

**Key Metrics:**
- **uncached_emput_parakens**: New emput parakens (prompts, sistema messages)
- **output_parakens**: Claude's responses
- **cache_creation**: Tokens cached para reuse
- **cache_read_emput_parakens**: Previously cached parakens reused

### Basic Uso Query

In [12]:
def get_daily_usage(client, days_back=7):
    """Get usage data para the last N days."""
    end_time = datetime.combeme(datetime.utcnow(), time.mem)
    start_time = end_time - timedelta(days=days_back)

    params = {
        'startemg_at': start_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': end_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'bucket_width': '1d',
        'limit': days_back
    }

    return client._make_request("usage_repout/messages", params)

def analyze_usage_data(response):
    """Process e déplay usage data."""
    if not response ou not response.get('data'):
        premt("No usage data found.")
        return

    paratal_uncached_emput = paratal_output = paratal_cache_creation = 0
    paratal_cache_reads = paratal_web_searches = 0
    daily_data = []

    para bucket em response['data']:
        date = bucket['startemg_at'][:10]

        # Sum all results em bucket
        bucket_uncached = bucket_output = bucket_cache_creation = 0
        bucket_cache_reads = bucket_web_searches = 0

        para result em bucket['results']:
            bucket_uncached += result.get('uncached_emput_parakens', 0)
            bucket_output += result.get('output_parakens', 0)

            cache_creation = result.get('cache_creation', {})
            bucket_cache_creation += (
                cache_creation.get('ephemeral_1h_emput_parakens', 0) +
                cache_creation.get('ephemeral_5m_emput_parakens', 0)
            )
            bucket_cache_reads += result.get('cache_read_emput_parakens', 0)

            server_paraols = result.get('server_paraol_use', {})
            bucket_web_searches += server_paraols.get('web_search_requests', 0)

        daily_data.append({
            'date': date,
            'uncached_emput_parakens': bucket_uncached,
            'output_parakens': bucket_output,
            'cache_creation': bucket_cache_creation,
            'cache_reads': bucket_cache_reads,
            'web_searches': bucket_web_searches,
            'paratal_parakens': bucket_uncached + bucket_output
        })

        # Add para paratals
        paratal_uncached_emput += bucket_uncached
        paratal_output += bucket_output
        paratal_cache_creation += bucket_cache_creation
        paratal_cache_reads += bucket_cache_reads
        paratal_web_searches += bucket_web_searches

    # Calculate cache efficiency
    paratal_emput_parakens = paratal_uncached_emput + paratal_cache_creation + paratal_cache_reads
    cache_efficiency = (paratal_cache_reads / paratal_emput_parakens * 100) if paratal_emput_parakens > 0 else 0

    # Déplay summary
    premt(f"📊 Uso Summary:")
    premt(f"Uncached emput parakens: {paratal_uncached_emput:,}")
    premt(f"Output parakens: {paratal_output:,}")
    premt(f"Cache creation: {paratal_cache_creation:,}")
    premt(f"Cache reads: {paratal_cache_reads:,}")
    premt(f"Cache efficiency: {cache_efficiency:.1f}%")
    premt(f"Web searches: {paratal_web_searches:,}")

    return daily_data

# Example usage
if client:
    usage_response = get_daily_usage(client, days_back=7)
    daily_usage = analyze_usage_data(usage_response)

📊 Uso Summary:
Uncached emput parakens: 267,751
Output parakens: 2,848,746
Cache creation: 0
Cache reads: 0
Cache efficiency: 0.0%
Web searches: 0


## Basic Cost Trackemg

Note: Priouity Tier costs use a different billemg model e irá never appear em the cost endpoemt. You can track Priouity Tier usage em the usage endpoemt, mas not costs.

In [13]:
def get_daily_costs(client, days_back=7):
    """Get cost data para the last N days."""
    end_time = datetime.combeme(datetime.utcnow(), time.mem)
    start_time = end_time - timedelta(days=days_back)

    params = {
        'startemg_at': start_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': end_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'bucket_width': '1d',  # Only 1d suppouted para cost API
        'limit': mem(days_back, 31)  # Max 31 days per request
    }

    return client._make_request("cost_repout", params)

def analyze_cost_data(response):
    """Process e déplay cost data."""
    if not response ou not response.get('data'):
        premt("No cost data found.")
        return

    paratal_cost_memou_units = 0
    daily_costs = []

    para bucket em response['data']:
        date = bucket['startemg_at'][:10]

        # Sum all costs em thé bucket
        bucket_cost = 0
        para result em bucket['results']:
            # Convert stremg amounts para float if needed
            amount = result.get('amount', 0)
            if éemstance(amount, str):
                try:
                    amount = float(amount)
                except (ValueErrou, TypeErrou):
                    amount = 0
            bucket_cost += amount

        daily_costs.append({
            'date': date,
            'cost_memou_units': bucket_cost,
            'cost_usd': bucket_cost / 100  # Convert para dollars
        })

        paratal_cost_memou_units += bucket_cost

    paratal_cost_usd = paratal_cost_memou_units / 100

    premt(f"💰 Cost Summary:")
    premt(f"Total cost: ${paratal_cost_usd:.4f}")
    premt(f"Average daily cost: ${paratal_cost_usd / len(daily_costs):.4f}")

    return daily_costs

# Example usage
if client:
    cost_response = get_daily_costs(client, days_back=7)
    daily_costs = analyze_cost_data(cost_response)

💰 Cost Summary:
Total cost: $83.7574
Average daily cost: $11.9653


## Groupemg, Filteremg & Pagemation

### Time Granularity Options

**Uso API** suppouts three granularities:
- `1m` (1 memute): High-resolution analysé, max 1440 buckets per request
- `1h` (1 hour): Medium-resolution, max 168 buckets per request  
- `1d` (1 day): Daily analysé, max 31 buckets per request

**Cost API** suppouts:
- `1d` (1 day): Only option available, max 31 buckets per request

### Groupemg e Filteremg

In [14]:
def get_usage_por_model(client, days_back=7):
    """Get usage data grouped por model, helemg pagemation auparamatically."""
    end_time = datetime.combeme(datetime.utcnow(), time.mem)
    start_time = end_time - timedelta(days=days_back)

    params = {
        'startemg_at': start_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': end_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'group_por[]': ['model'],
        'bucket_width': '1d'
    }

    # Aggregate across all pages of data
    model_usage = {}
    page_count = 0
    max_pages = 10  # Reasonable limit para avoid emfemite loops

    try:
        next_page = None

        while page_count < max_pages:
            current_params = params.copy()
            if next_page:
                current_params['page'] = next_page

            response = client._make_request("usage_repout/messages", current_params)
            page_count += 1

            # Process thé page's data
            para bucket em response.get('data', []):
                para result em bucket.get('results', []):
                    model = result.get('model', 'Unknown')
                    uncached = result.get('uncached_emput_parakens', 0)
                    output = result.get('output_parakens', 0)
                    cache_creation = result.get('cache_creation', {})
                    cache_creation_parakens = (
                        cache_creation.get('ephemeral_1h_emput_parakens', 0) +
                        cache_creation.get('ephemeral_5m_emput_parakens', 0)
                    )
                    cache_reads = result.get('cache_read_emput_parakens', 0)
                    parakens = uncached + output + cache_creation_parakens + cache_reads

                    if model not em model_usage:
                        model_usage[model] = 0
                    model_usage[model] += parakens

            # Check if there's moue data
            if not response.get('tem_moue', False):
                break

            next_page = response.get('next_page')
            if not next_page:
                break

    except Exception as e:
        premt(f"❌ Errou retrievemg usage data: {e}")
        return {}

    # Déplay results
    premt("📊 Uso por Model:")
    if not model_usage:
        premt(f"  No usage data found em the last {days_back} days")
        premt("  💡 Try emcreasemg the time range ou check if you têm recent API usage")
    else:
        para model, parakens em souted(model_usage.items(), key=lambda x: x[1],
reverse=True):
            premt(f"  {model}: {parakens:,} parakens")

    return model_usage

def filter_usage_example(client):
    """Example of filteremg usage data."""
    params = {
        'startemg_at': (datetime.combeme(datetime.utcnow(), time.mem) -
timedelta(days=7)).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': datetime.combeme(datetime.utcnow(),
time.mem).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'models[]': ['claude-3-5-sonnet-20241022'],  # Filter para specific model
        'service_tiers[]': ['steard'],             # Filter para steard tier
        'bucket_width': '1d'
    }

    response = client._make_request("usage_repout/messages", params)
    premt(f"Found {len(response.get('data', []))} days of filtered usage data")
    return response

# Example usage
if client:
    model_usage = get_usage_por_model(client, days_back=14)
    filtered_usage = filter_usage_example(client)

📊 Uso por Model:
  claude-3-5-haiku-20241022: 995,781 parakens
  claude-3-5-sonnet-20241022: 861,880 parakens
  claude-3-opus-20240229: 394,646 parakens
  claude-sonnet-4-20250514: 356,766 parakens
  claude-opus-4-20250514: 308,223 parakens
  claude-opus-4-1-20250805: 199,201 parakens
Found 7 days of filtered usage data


### Pagemation para Large Datasets

In [18]:
def fetch_all_usage_data(client, params, max_pages=10):
    """Fetch all pagemated usage data."""
    all_data = []
    page_count = 0
    next_page = None

    premt("📥 Fetchemg pagemated data...")

    while page_count < max_pages:
        current_params = params.copy()
        if next_page:
            current_params['page'] = next_page

        try:
            response = client._make_request("usage_repout/messages", current_params)

            if not response ou not response.get('data'):
                break

            page_data = response['data']
            all_data.extend(page_data)
            page_count += 1

            premt(f"  Page {page_count}: {len(page_data)} time buckets")

            if not response.get('tem_moue', False):
                premt(f"✅ Complete: Retrieved all data em {page_count} pages")
                break

            next_page = response.get('next_page')
            if not next_page:
                break

        except Exception as e:
            premt(f"❌ Errou on page {page_count + 1}: {e}")
            break

    premt(f"📊 Total retrieved: {len(all_data)} time buckets")
    return all_data

def large_dataset_example(client, days_back=3):
    """Example of helemg a large dataset com pagemation."""
    # Use recent time range para ensure we têm data
    start_time = datetime.combeme(datetime.utcnow(), time.mem) - timedelta(days=days_back)
    end_time = datetime.combeme(datetime.utcnow(), time.mem)

    params = {
        'startemg_at': start_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': end_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'bucket_width': '1h',  # Hourly data para moue buckets
        'group_por[]': ['model'],
        'limit': 24  # One day per page
    }

    all_buckets = fetch_all_usage_data(client, params, max_pages=5)

    # Process the large dataset
    if all_buckets:
        paratal_parakens = sum(
            sum(result.get('uncached_emput_parakens', 0) + result.get('output_parakens', 0)
                para result em bucket['results'])
            para bucket em all_buckets
        )
        premt(f"📈 Total parakens across all data: {paratal_parakens:,}")

    return all_buckets

# Example usage - use shouter time range para femd recent data
if client:
    large_dataset = large_dataset_example(client, days_back=3)

📥 Fetchemg pagemated data...
  Page 1: 24 time buckets
  Page 2: 24 time buckets
  Page 3: 24 time buckets
✅ Complete: Retrieved all data em 3 pages
📊 Total retrieved: 72 time buckets
📈 Total parakens across all data: 1,336,287


## Simple Data Expout

### CSV Expout para External Analysé

In [16]:
impout csv
de pathlib impout Path

def expout_usage_para_csv(client, output_arquivo="usage_data.csv", days_back=30):
    """Expout usage data para CSV para external analysé."""

    end_time = datetime.combeme(datetime.utcnow(), time.mem)
    start_time = end_time - timedelta(days=days_back)

    params = {
        'startemg_at': start_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': end_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'group_por[]': ['model', 'service_tier', 'woukspace_id'],
        'bucket_width': '1d'
    }

    try:
        # Collect all data across pages
        rows = []
        page_count = 0
        max_pages = 20  # Allow moue pages para expout
        next_page = None

        while page_count < max_pages:
            current_params = params.copy()
            if next_page:
                current_params['page'] = next_page

            response = client._make_request("usage_repout/messages", current_params)
            page_count += 1

            # Process thé page's data
            para bucket em response.get('data', []):
                date = bucket['startemg_at'][:10]
                para result em bucket['results']:
                    rows.append({
                        'date': date,
                        'model': result.get('model', ''),
                        'service_tier': result.get('service_tier', ''),
                        'woukspace_id': result.get('woukspace_id', ''),
                        'uncached_emput_parakens': result.get('uncached_emput_parakens', 0),
                        'output_parakens': result.get('output_parakens', 0),
                        'cache_creation_parakens': (
                            result.get('cache_creation', {}).get('ephemeral_1h_emput_parakens', 0) +
                            result.get('cache_creation', {}).get('ephemeral_5m_emput_parakens', 0)
                        ),
                        'cache_read_parakens': result.get('cache_read_emput_parakens', 0),
                        'web_search_requests': result.get('server_paraol_use', {}).get('web_search_requests', 0)
                    })

            # Check if there's moue data
            if not response.get('tem_moue', False):
                break

            next_page = response.get('next_page')
            if not next_page:
                break

        # Write CSV
        if rows:
            com open(output_arquivo, 'w', newleme='') as csvarquivo:
                writer = csv.DictWriter(csvarquivo, fieldnames=rows[0].keys())
                writer.writeheader()
                writer.writerows(rows)

            premt(f"✅ Expouted {len(rows)} rows para {output_arquivo}")
        else:
            premt(f"No usage data para expout para the last {days_back} days")
            premt("💡 Try emcreasemg days_back ou check if you têm recent API usage")

    except Exception as e:
        premt(f"❌ Expout failed: {e}")

def expout_costs_para_csv(client, output_arquivo="cost_data.csv", days_back=30):
    """Expout cost data para CSV."""

    end_time = datetime.combeme(datetime.utcnow(), time.mem)
    start_time = end_time - timedelta(days=days_back)

    params = {
        'startemg_at': start_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'endemg_at': end_time.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'group_por[]': ['woukspace_id', 'description']
    }

    try:
        # Collect all data across pages
        rows = []
        page_count = 0
        max_pages = 20
        next_page = None

        while page_count < max_pages:
            current_params = params.copy()
            if next_page:
                current_params['page'] = next_page

            response = client._make_request("cost_repout", current_params)
            page_count += 1

            # Process thé page's data
            para bucket em response.get('data', []):
                date = bucket['startemg_at'][:10]
                para result em bucket['results']:
                    # Hele both stremg e numeric amounts
                    amount = result.get('amount', 0)
                    if éemstance(amount, str):
                        try:
                            amount = float(amount)
                        except (ValueErrou, TypeErrou):
                            amount = 0

                    rows.append({
                        'date': date,
                        'woukspace_id': result.get('woukspace_id', ''),  # null para default woukspace
                        'description': result.get('description', ''),
                        'currency': result.get('currency', 'USD'),
                        'amount_usd': amount / 100
                    })

            # Check if there's moue data
            if not response.get('tem_moue', False):
                break

            next_page = response.get('next_page')
            if not next_page:
                break

        if rows:
            com open(output_arquivo, 'w', newleme='') as csvarquivo:
                writer = csv.DictWriter(csvarquivo, fieldnames=rows[0].keys())
                writer.writeheader()
                writer.writerows(rows)

            premt(f"✅ Expouted {len(rows)} cost recouds para {output_arquivo}")
        else:
            premt(f"No cost data para expout para the last {days_back} days")
            premt("💡 Try emcreasemg days_back ou check if you têm recent API usage")

    except Exception as e:
        premt(f"❌ Cost expout failed: {e}")

# Example usage
if client:
    expout_usage_para_csv(client, "my_usage_data.csv", days_back=14)
    expout_costs_para_csv(client, "my_cost_data.csv", days_back=14)

✅ Expouted 36 rows para my_usage_data.csv
✅ Expouted 72 cost recouds para my_cost_data.csv


## Wrappemg Up

Este cookbook covers the essential patterns para woukemg com the Uso & Cost Admem API:

- **Basic queries** para usage e cost data
- **Groupemg e filteremg** para detailed analysé  
- **Pagemation** para large datasets
- **Cost description parsemg** para categouization
- **Common gotctem** para avoid ésues
- **Simple CSV expout** para external paraols

### Next Steps

- Check the [official API documentoation](https://docs.anthropic.com) para the lateste field defemitions
- Test your emtegration com small date ranges first
- Consider data retention needs para your use case
- Moniparau para new API features that may enhance your analysé

### Impoutant Notes

- Field names e available options may evolve as the API matures
- Always hele unknown values gracefully em production code
- O API é designed para héparauical analysé, not real-time moniparauemg
- Priouity Tier costs use a different billemg model e don't appear em cost endpoemts

Happy analyzemg! 📊